# ODI to Databricks Migration

**Source File:** `TARGET.sql`
**Conversion Timestamp:** 2024-07-30T12:00:00Z
**Description:** Basic insert into a target department table.

In [ ]:
import datetime

dbutils.widgets.text("ETL_JOB_TYPE", "", "1. ETL Job Type (e.g., 'FULL', 'INCREMENTAL')")
dbutils.widgets.text("DATASOURCE_NUM_ID", "-1", "2. Data Source Number ID")
dbutils.widgets.text("ETL_PROC_WID", "-1", "3. ETL Process Widget")
dbutils.widgets.text("ODI_SESS_NO", "-1", "4. ODI Session Number")
dbutils.widgets.text("ETL_LAST_EXTRACT_TIME", datetime.datetime(1900, 1, 1).strftime('%Y-%m-%d %H:%M:%S'), "5. Last Extract Time (YYYY-MM-DD HH:MI:SS)")
dbutils.widgets.text("ETL_CURRENT_EXTRACT_TIME", datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "6. Current Extract Time (YYYY-MM-DD HH:MI:SS)")

## ETL Parameters

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_parameters AS
SELECT
  '${ETL_JOB_TYPE}' AS etl_job_type,
  CAST('${DATASOURCE_NUM_ID}' AS BIGINT) AS datasource_num_id,
  CAST('${ETL_PROC_WID}' AS BIGINT) AS etl_proc_wid,
  CAST('${ODI_SESS_NO}' AS BIGINT) AS odi_sess_no,
  to_timestamp('${ETL_LAST_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_last_extract_time,
  to_timestamp('${ETL_CURRENT_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_current_extract_time;

In [ ]:
display(spark.sql("SELECT * FROM v_etl_parameters"))

## Insert into Target Table

In [ ]:
%sql
-- SCEN_TASK_NO in {10}
-- SCEN_TASK_NO in {20}
-- SCEN_TASK_NO in {30}
INSERT INTO workspace.hr.trg_dep (
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID
)
SELECT
    departments.DEPARTMENT_ID,
    departments.DEPARTMENT_NAME,
    departments.MANAGER_ID,
    departments.LOCATION_ID
FROM
    workspace.hr.departments AS departments;

## Validation

In [ ]:
%sql
SELECT COUNT(*) AS total_records FROM workspace.hr.trg_dep;

## Conversion Notes and Manual Actions Required

1.  **Schema and Table Naming:** Oracle schema `HR` was converted to `workspace.hr`. Table names `TRG_DEP` and `DEPARTMENTS` were converted to lowercase `trg_dep` and `departments` respectively, prefixed with `workspace.hr`.
2.  **Oracle Hints Removed:** The `/*+ APPEND PARALLEL */` hint was removed as it is specific to Oracle and not applicable in Databricks Spark SQL for Delta tables.
3.  **Column Casing:** Column names (`DEPARTMENT_ID`, `DEPARTMENT_NAME`, `MANAGER_ID`, `LOCATION_ID`) were preserved as-is from the source, respecting case.
4.  **Assumptions:**
    *   It is assumed that the target table `workspace.hr.trg_dep` already exists with a compatible schema (e.g., `DEPARTMENT_ID` as `BIGINT`, `DEPARTMENT_NAME` as `STRING`, `MANAGER_ID` as `BIGINT`, `LOCATION_ID` as `BIGINT`). If not, a `CREATE TABLE` DDL for `trg_dep` based on the source `DEPARTMENTS` table's schema would be required before this `INSERT`.
    *   No specific data type conversions were required for the `SELECT` statement itself, as Spark SQL handles basic type inference and compatibility. If the source Oracle types were known (e.g., `NUMBER(p,s)`, `VARCHAR2`), explicit casting might be added for robustness.
5.  **Parameter Widgets:** Placeholder widgets have been added for `ETL_JOB_TYPE`, `DATASOURCE_NUM_ID`, `ETL_PROC_WID`, `ODI_SESS_NO`, `ETL_LAST_EXTRACT_TIME`, and `ETL_CURRENT_EXTRACT_TIME` to conform to the standard notebook structure, although they are not directly used in this specific SQL statement. The `v_etl_parameters` view is created to demonstrate how to expose these widgets as a SQL view.